# The readout that belongs to nobody

A house runs experiments for eleven parties. In the eleventh month somebody pools nine
studies of the same treatment and gets a tighter interval than any of them. It is a good
number. It is also wrong, and the reason is not in any of the nine records: one of them
came out of another party's programme and was never licensed to travel, and two of them
were read four weeks later than the plan said, because the field ran long.

Inside one analysis `axiom` will not let either of those happen quietly. A window
difference is a `TransferPlan` with a status and a named assumption; a unit conversion
writes a `LedgerLine`. Between analyses there was nothing: `ArtifactRegistry` is a flat
`root/<hash>.json` that cannot say *whose* artifact it holds, and `StudyBuilder` builds a
plan and later builds a `Measurement` without ever comparing the two ends.

This notebook is the two objects that close those gaps — `Catalog`, which gives every
artifact a scope, and `ExperimentRun`, which binds a plan to the readout that answers it.

In [ ]:
import tempfile
from pathlib import Path

from axiom.core import D, Intervention, LedgerLine, Outcome, Population, TimeWindow, Treatment
from axiom.design import pulse
from axiom.estimands import Estimand, Level, Quantity
from axiom.io import (
    Catalog,
    CatalogEntry,
    CrossScopeError,
    Deviation,
    ExperimentRun,
    LifecycleError,
    Program,
    ProgramStore,
    Stage,
    Transition,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way

enable();  # every axiom result renders itself

## A programme is a scope

`Program` is a `party` and a line of work within it. The vocabulary is general on purpose:
a commercial house reads `party` as the client, a trial as the site, a consortium as the
member. Both halves are path-safe tokens, so `scope` is a two-component key that cannot
climb out of the catalog root — `Program(party="../etc", ...)` does not construct.

In [ ]:
catalog = Catalog(Path(tempfile.mkdtemp()) / "catalog")

northwind = Program(party="northwind", program="dose-response", started="2026-01-05",
                    description="fertilizer dose-response across the north region")
acme = Program(party="acme", program="dose-response", started="2026-02-11")
for p in (northwind, acme):
    catalog.register(p)

table([[p.scope, p.started or "—", p.description or "—"] for p in catalog.programs()],
      headers=("scope", "started", "description"), title="registered programmes")

try:
    Program(party="../etc", program="passwd")
except ValueError as e:
    print("refused:", next(ln.strip() for ln in str(e).splitlines() if "scope token" in ln))

## An artifact belongs to exactly one scope

`Catalog.store(program)` returns a `ProgramStore` — the registry surface, restricted. What
matters is the read side: asking for another party's artifact does not return `None` and
does not return the artifact. It raises `CrossScopeError` naming both scopes, because *no
such artifact* and *that is somebody else's artifact* are different facts, and the second
one is the one worth an alert.

In [ ]:
north: ProgramStore = catalog.store(northwind)
south: ProgramStore = catalog.store(acme)

season = TimeWindow(start=0, stop=8, basis="cumulative")
schedule = pulse(8, 100.0, 0.0, on=3, off=2, treatment="fertilizer")

window_hash = north.put(season, label="window", tags={"role": "plan"})
schedule_hash = north.put(schedule, label="schedule", tags={"role": "plan"})
print(north, "at", north.root.name)
print("scope of the window:", catalog.scope_of(window_hash))

try:
    south.get(window_hash)
except CrossScopeError as e:
    print("\nCrossScopeError:", e)

try:
    south.get("0" * 64)
except KeyError as e:
    print("KeyError:      ", e)

## Crossing the boundary is a transfer, and a transfer takes a ledger line

Sharing across parties is a real and defensible move — a pooled prior from one programme
informing another's model is the whole point of `meta`. It is also a move somebody has to
be able to find six months later. `Catalog.transfer` is the only route, the `LedgerLine` is
not optional, and the line is stored in the target scope so `crossings()` is the list.

In [ ]:
line = LedgerLine(
    kind="scope_crossing",
    statement="northwind's season window is reused for acme under the shared-methodology agreement",
    detail={"agreement": "DPA-14"},
)
line_hash = catalog.transfer(window_hash, source=northwind, target=acme, line=line)

print("acme can now read it:", south.get(window_hash) == season)
print("held by:", catalog.scope_of(window_hash))
table([[e.scope, e.label, e.derived_from[:12] + "…"] for e in catalog.crossings()],
      headers=("into scope", "label", "licensed by"), title="every artifact that crossed a boundary")
print("the licence itself:", south.get(line_hash).statement)

## The index carries columns, because a scan is not a query

The flat registry's `index.tsv` was `hash\ttype`: it could say what was in the root and
had to open every file to learn anything else. A `CatalogEntry` is a row with a scope, a
label, a timestamp, the licence that let it cross, and free tags — and `find` filters on
all of them without opening a single artifact.

In [ ]:
rows = catalog.find(type_name="TimeWindow")
table([[e.scope, e.label, e.type_name.split(":")[-1], e.tags.get("role", "—")] for e in rows],
      headers=("scope", "label", "type", "role"), title='find(type_name="TimeWindow")')

print("plan artifacts in northwind:", len(north.find(tags={"role": "plan"})))
print("artifacts in the catalog:   ", len(catalog), "across", catalog.scopes())

entry = north.find(label="schedule")[0]
print("\na row round-trips through the TSV:", CatalogEntry.from_row(entry.row()) == entry)

## The other axis: the plan, and the readout that claims to answer it

Now the second gap. An `ExperimentRun` holds the roles a plan is made of — by content hash,
which is what lets it live in `io` below everything it records — and moves through
`designed → committed → running → read → calibrated → pooled`. `commit` is the transition
that matters: it freezes `plan_hash` over the roles as they stand.

In [ ]:
lift = Estimand(
    name="lift_at_100",
    quantity=Quantity(kind="contrast"),
    treatment=Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
    intervention=Intervention(doses={"fertilizer": 100.0}, version="granular"),
    reference=Intervention(doses={"fertilizer": 0.0}, version="granular"),
    outcome=Outcome(name="yield_total", dimension=D.outcome, unit="kg", aggregation="sum"),
    population=Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}}),
    window=season,
    level=Level(unit="cluster", interference="none"),
    dimension=D.outcome,
    description="season-total yield lift from 100 USD of granular fertilizer vs none",
)
north.put(lift, label="estimand", tags={"role": "plan"})

planned = ExperimentRun.design(
    "NW-14", scope=northwind.scope, estimand=lift,
    roles={"window": season, "schedule": schedule},
    at="2026-03-02T09:00:00+00:00",
).commit(at="2026-03-04T09:00:00+00:00")

print(planned.stage, "| plan", planned.plan_hash[:16])
print("conformance before the readout:", planned.conformance().status)

## A clean run

The readout attaches as a hash and the estimand it answers attaches with it. Nothing moved
between commit and read, so `conformance()` is `identified` — the strongest of the four
answers, and a claim the object can actually check rather than one a person remembers
making.

In [ ]:
measurement_hash = "9f" * 32  # in practice, calibrate.Measurement(...).content_hash()

clean = (planned
         .start(at="2026-03-09T09:00:00+00:00")
         .read(measurement_hash, estimand=lift, at="2026-05-04T09:00:00+00:00"))

print(clean.summary())
table([[t.stage, t.at[:10], t.note or "—"] for t in clean.history],
      headers=("stage", "on", "note"), title="NW-14, as it happened")

## The field ran four weeks long

Which is what actually happens. `deviate` moves the role *and* records what moved, from
what, to what, and why. The verdict drops to `downgraded` and carries the departure as an
`Assumption` with its own `challenged_by` — the same shape `identify` and `calibrate`
already use, so a conformance verdict travels into `meta` as a fact about the number rather
than a footnote beside it.

In [ ]:
long_season = TimeWindow(start=0, stop=12, basis="cumulative")

ran_long = (planned
            .start(at="2026-03-09T09:00:00+00:00")
            .deviate("window", long_season, reason="the field ran four weeks long",
                     at="2026-05-04T09:00:00+00:00")
            .read(measurement_hash, estimand=lift, at="2026-06-01T09:00:00+00:00"))

verdict = ran_long.conformance()
print(verdict.status, "—", verdict.reason)

deviation: Deviation = ran_long.deviations[0]
assumption = deviation.assumption()
table([["role", deviation.role],
       ["planned", deviation.planned[:16] + "…"],
       ["realized", deviation.realized[:16] + "…"],
       ["reason", deviation.reason],
       ["state", assumption.state],
       ["challenged by", assumption.challenged_by]],
      headers=("", "the departure, as recorded"))

## An unrecorded change is worse than a recorded one

This is the invariant with teeth. A committed plan whose roles no longer hash to
`plan_hash`, with no deviation filed, is not `downgraded` — it is `blocked`. So is a readout
that answers a different estimand than the one committed, which is the nine-studies failure
at the top of this notebook. And a stage that cannot follow the current one raises rather
than quietly recording a life the run did not have.

In [ ]:
tampered = planned.model_copy(update={"roles": {**planned.roles, "window": long_season.content_hash()}})
unrecorded = tampered.start(at="t").read(measurement_hash, estimand=lift, at="t")

wrong_quantity = planned.start(at="t").read(
    measurement_hash, estimand=lift.model_copy(update={"name": "lift_at_200"}), at="t")

never_committed = (ExperimentRun.design("NW-15", roles={"window": season}, estimand=lift)
                   .model_copy(update={"stage": "running"})
                   .read(measurement_hash, estimand=lift, at="t"))

table([[name, v.status, v.reason or "—"] for name, v in [
        ("read as committed", clean.conformance()),
        ("deviation filed", ran_long.conformance()),
        ("plan changed, nothing filed", unrecorded.conformance()),
        ("answers another estimand", wrong_quantity.conformance()),
        ("read with no committed plan", never_committed.conformance()),
        ("still in the field", planned.start(at="t").conformance()),
      ]], headers=("the run", "conformance", "why"), title="the four answers")

try:
    planned.pooled()
except LifecycleError as e:
    print("\nLifecycleError:", e)

## Filed where it belongs

A run is a `Spec` like any other, so it goes in its programme's store and comes back out of
the index by stage. `Stage` is the vocabulary, and every `Transition` on the run says when
it happened. Six months later the question "is this readout the analysis we said we would
run?" is answered by the artifact, not by whoever still remembers.

In [ ]:
for run in (planned, clean, ran_long):
    north.put(run, label=run.experiment, tags={"stage": run.stage})

stages: list[Stage] = ["designed", "committed", "running", "read", "calibrated", "pooled"]
found = north.find(type_name="ExperimentRun")
table([[e.label, e.tags["stage"], e.digest[:12] + "…"] for e in found],
      headers=("experiment", "stage", "digest"), title="NW-14, every version filed")

reloaded = north.get(found[-1].digest)
last: Transition = reloaded.history[-1]
print(reloaded, "| last transition:", last.stage, "on", last.at[:10])
print("stages a run may reach:", ", ".join(stages))

## What does this party mean by "conversion"?

The scope tells you whose artifact it is and the run tells you whether the readout was the
analysis anybody planned. Neither says what the number is *of*. A `StudyRecord` carries a
quantity name; an `Estimand` names a `core.Outcome`; and nothing has ever compared one party's
`Outcome` to another's, because the comparison lives in a `Spec` that neither record carries.

Two things go wrong, and they are different. Party A's "conversion" counts a trial signup and
party B's counts a paid one — **disagreement**. Or one party changes its mind in week nine and
the records on either side of the change are pooled without comment — **drift**.

A `DefinitionRegistry` names, versions and content-hashes whatever `Spec` a party has agreed a
term means, so both questions have answers instead of opinions. It is composed over the same
`Catalog`, so definitions live in their party's scope like everything else.

In [ ]:
from axiom.core import Outcome
from axiom.io import Change, Consensus, Definition, DefinitionRegistry, registered

acme_growth = Program(party="acme", program="growth")
globex_growth = Program(party="globex", program="growth")
northwind_growth = Program(party="northwind", program="growth")
for p in (acme_growth, globex_growth, northwind_growth):
    catalog.register(p)

definitions = DefinitionRegistry(catalog)
paid = Outcome(name="conversion", dimension=D.outcome, unit="count", aggregation="sum")
rate = Outcome(name="conversion", dimension=D.outcome, unit="count", aggregation="mean")

first: Definition = definitions.register(northwind_growth, "conversion", paid, at="2026-01-05")
same = definitions.register(northwind_growth, "conversion", paid, at="2026-01-19")
print("registering the same spec again is a no-op:", first == same, "| version", same.version)
print(first)

### Drift: a change is a diff, not a flag

Versions are content, not intent. Registering an identical spec returns the version that
already exists; registering a different one is `n + 1`, with `supersedes` pointing at the
digest it replaced. Nobody has to remember to bump anything and nobody can bump without
changing something.

So "somebody redefined conversion in week nine" comes back as the field that moved and the
date it moved, which is the sentence an analyst can act on.

In [ ]:
definitions.register(northwind_growth, "conversion", rate, at="2026-03-02",
                     note="growth team switched to a per-visit rate")
table([[d.version, d.registered, d.digest[:12] + "…", d.supersedes[:12] + "…" if d.supersedes else "—",
        d.note or "—"] for d in definitions.history(northwind_growth, "conversion")],
      headers=("version", "registered", "digest", "supersedes", "note"),
      title="northwind/growth: conversion")

for change in definitions.changes(northwind_growth, "conversion"):
    assert isinstance(change, Change)
    print(f"{change.at}: v{change.from_version} -> v{change.to_version}")
    for field, moved in sorted(change.changed.items()):
        print(f"   {field}: {moved}")

### Disagreement: silence is not agreement

`consensus` asks whether several parties currently mean the same thing. Three answers, and the
third is the one that matters: a party that has **never registered** the term has not agreed
with anybody, so the verdict is `unverified` rather than `identified`. Counting silence as
agreement is the exact failure the registry exists to prevent.

In [ ]:
definitions.register(acme_growth, "conversion", paid, at="2026-01-05")
quiet: Consensus = definitions.consensus(
    "conversion", [northwind_growth, acme_growth, globex_growth])
print(quiet.summary())
print(" verdict:", quiet.verdict().status, "—", quiet.verdict().reason[:90])

definitions.register(globex_growth, "conversion", paid, at="2026-01-05")
disagreed = definitions.consensus("conversion", [northwind_growth, acme_growth, globex_growth])
print("\n" + disagreed.summary())
print(" verdict:", disagreed.verdict().status)
print(" ", disagreed.verdict().reason[:170])

In [ ]:
definitions.register(northwind_growth, "conversion", paid, at="2026-04-01", note="reverted")
agreed = definitions.consensus("conversion", [northwind_growth, acme_growth, globex_growth])
print("after northwind reverts:", agreed.verdict().status,
      "| agreed on", agreed.agreed[:12] + "…")
print("northwind is now at version", definitions.current(northwind_growth, "conversion").version,
      "and its history is still", [d.version for d in definitions.history(northwind_growth, "conversion")])
table([[name, d.version, d.type_name.split(":")[-1], d.digest[:12] + "…"]
       for name, d in registered(definitions, northwind_growth).items()],
      headers=("term", "current version", "type", "digest"),
      title="everything northwind/growth has defined")

Three artifacts and three rules. Every artifact has a scope, and leaving it is a transfer with
a ledger line. Every term a party uses has a versioned definition, and changing it is a diff
with a date. Every readout has a committed plan behind it, and departing from it is a
deviation with a reason — with the difference between *recorded* and *unrecorded* carried in
the same four words identification already uses.

That does not make the nine pooled studies right. It makes the two that were read late say
so, and the one that came from another party findable in `crossings()`. What it buys is the
premise every cross-party comparison was already resting on, written down where it can be
checked. The rest of the programme-level backlog — assignment and delivery checks,
collision, a party level in `meta.pool`, program-wide error control — is
`docs/notes/0027-scope-and-the-experiment-lifecycle.md`.